### Setting the topology

In a YAML file, define the parameters and topology for your test in a similar manner as the following:
```yaml
topology:
  name: "g5k_mcast_eval"
  wall_time: "2hr"
  relay_nodes: false # whether to add one relay machine in each cluster
  netns_per_client: 5 # number of network namespaces to run on each client
  # see the possible frrouting version at https://deb.frrouting.org/
  frrouting_version: "frr-10.4"
  router_template: "base_router_config_ospf.frr" # path to the router configuration template

  server:
    cluster: "chirop" # Lille
    nodes: 1
    # node: "chirop-5.lille.grid5000.fr"   # optional: pin a specific machine

  # each site has one router + num_clients clients and one relay,
  # all reserved in the given cluster. 
  sites:
    - name: nancy
      cluster: gros
      number: 5
    - name: rennes
      cluster: parasilo
      number: 5
    - name: nantes
      cluster: ecotype
      number: 5
    - name: lyon
      cluster: nova
      number: 5

  # links are established between routers in different clusters
  # GRE tunnels are established between the two routers, with OSPF running over it.
  # endpoints must be router_server or router_client_CLIENT-CLUSTER-ID
  links:
    - [router_server, router_client_0]             # src -> nancy
    - [router_client_0, router_client_1]           # nancy -> rennes
    - [router_client_0, router_client_3]           # nancy -> lyon
    - [router_client_0, router_client_2]           # rennes -> nantes
``` 


In [1]:
# !pip install enoslib ipywidgets==8.1.5 fabric --break-system-packages
!pip install -U jupyterlab ipywidgets jupyterlab-widgets --break-system-packages

Defaulting to user installation because normal site-packages is not writeable



### Setting up the experiment
Once you have your `topology.yaml` file, you can create the `G5KExpe` class which will handle most things for you.

In [2]:
from gem import G5KExpe

experiment = G5KExpe(
    # change the path to point to your topology yaml file
    topology_conf="./mcast_eval.yaml",
    #
    # other parameters exist:
    # g5k_conf_file_loc points to your .python-grid5000.yaml file which contains your grid5000 credentials, by default it is in `~/` (so `/home/USERNAME`)
    # g5k_conf_file_loc=".python-grid5000.yaml"
    #
    # job_type should be deploy, but you may need it to be different
    # job_type="deploy"
    #
    # os_env_name defines the OS environement that is deployed on the machines
    # by default it is debian12 with NFS, however you can find the entire list at https://www.grid5000.fr/w/Getting_Started#:~:text=On%20Grid%275000%20reference%20environments
    # Make sure to pick debian to ensure that the packages are properly installed
    # os_env_name="debian12-nfs"
    #
    # You can configure the number of ansible forks used, ansible's default is 5, meaning that it'll run commands on at most 5 host at once
    # in this framework the default is 25 to make use of more parallelism, however, increasing this value will consume more resources (especially memory)
    # setting the number of forks to 200 will consume around 25 GB of memory but will allow ansible to perform operations on 200 hosts at the same time
    ansible_forks=30,
)

# you should always follow grid5000's usage policy (see https://www.grid5000.fr/w/Grid5000:UsagePolicy)
# this method simply checks that the job you are trying to start will not cross the day-night boundary.
# If it does, it'll warn you. You can always comment this out if you wish...
experiment.usage_policy_check()

provider = experiment.setup_enoslib_conf()

[WARNING]: failed to patch stdout/stderr for fork-safety: 'OutStream' object
has no attribute 'buffer'
[WARNING]: failed to reconfigure stdout/stderr with custom encoding error
handler: 'OutStream' object has no attribute 'reconfigure'


_____        ___  ____  _ _ _
 | ____|_ __  / _ \/ ___|| (_) |__
 |  _| | '_ \| | | \___ \| | | '_ \
 | |___| | | | |_| |___) | | | |_) |
 |_____|_| |_|\___/|____/|_|_|_.__/  10.9.0

 • Documentation: ]8;id=87294;https://discovery.gitlabpages.inria.fr/enoslib/\https://discovery.gitlabpages.inria.fr/enoslib/]8;;\                            
 • Source: ]8;id=303356;https://gitlab.inria.fr/discovery/enoslib\https://gitlab.inria.fr/discovery/enoslib]8;;\                                         
 • Chat: ]8;id=520843;https://framateam.org/enoslib\https://framateam.org/enoslib]8;;\

                         Dependency check                         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider      ┃    Status     ┃ Hint                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Chameleon     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonKVM  │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonEdge │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Fabric        │ NOT INSTALLED │ pip install enoslib[fabric]    │
│ Distem        │ NOT INSTALLED │ pip install enoslib[distem]    │
│ IOT-lab       │ NOT INSTALLED │ pip install enoslib[iotlab]    │
│ Grid'5000     │   INSTALLED   │                                │
│ Openstack     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Vagrant       │ NOT INSTALLED │ pip install enoslib[vagrant]   │
│ VMonG5k       │   INSTALLED   │                                │
└───────────────┴───────────────┴────────────────────────────────┘

                                Connectivity check                                 
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider  ┃ Key                 ┃ Connectivity ┃ Hint                           ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Grid'5000 │ ssh:access          │      ✅      │ Connection to access.grid5000… │
│ Grid'5000 │ ssh:access:frontend │      ✅      │ Connection Host(rennes.grid50… │
│ Grid'5000 │ api:access          │      ✅      │                                │
│ VMonG5k   │ access              │      ❔      │ Check G5k status               │
└───────────┴─────────────────────┴──────────────┴────────────────────────────────┘


### Reserving resources
Now that G5K is setup, we can create the experiment's reservation by defining the number of machines of each role and in each cluster.

Once done, we proceed with the actual reservation of the machines. Be aware that this step may take some time (minimum 5 minutes). This is due to the deployment of the VM image. 

**Don't forget to run "ssh-add KEY_PATH" to allow ansible to connect using your ssh key**

In [3]:
!pip install -U jupyterlab ipywidgets jupyterlab-widgets --break-system-packages
experiment.reserve_res(provider)
display(experiment.roles)

Defaulting to user installation because normal site-packages is not writeable
Reserving resources now, might take a while...


INFO     [G5k] Reloading 2209758 from lille                              ]8;id=849331;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=235081;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 4126993 from rennes                             ]8;id=748138;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=462427;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 2068926 from lyon                               ]8;id=603062;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=769027;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 338507 from nantes                              ]8;id=965335;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=249484;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

...

Output()

Results : []


Finished 5 tasks (Gather facts,Ensure apt keyring directory exists,Download FRR GPG key,Add 
FRR apt repository,Install FRR packages) on {'parasilo-25.rennes.grid5000.fr', 
'ecotype-43.nantes.grid5000.fr', 'nova-18.lyon.grid5000.fr', 'gros-9.nancy.grid5000.fr', 
'chirop-4.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

ip
127.0.0.1/8 # noqa
::1/128 # noqa
ip
fe80::a236:9fff:fed2:a250/64 # noqa
172.16.52.18/20 # noqa
ip
127.0.0.1/8 # noqa
::1/128 # noqa
ip
172.16.33.4/20 # noqa


**Optional**: You can refresh the roles by running the cell below (useful when the notebook closed but you have a reservation running)

In [ ]:
experiment.sync_info()

#### Setting up interfaces, IP subnets, and Network namespaces

In [4]:
experiment.setup_interfaces()
experiment.assign_node_ips()
experiment.netns_setup_macvlan()

Output()

Prod interface for nova-8.lyon.grid5000.fr: enp5s0f0
Prod interface for gros-90.nancy.grid5000.fr: eno1
Prod interface for gros-91.nancy.grid5000.fr: eno1
Prod interface for ecotype-5.nantes.grid5000.fr: eno1
Prod interface for parasilo-25.rennes.grid5000.fr: eno1
Prod interface for gros-96.nancy.grid5000.fr: eno1
Prod interface for nova-9.lyon.grid5000.fr: enp5s0f0
Prod interface for ecotype-7.nantes.grid5000.fr: eno1
Prod interface for nova-21.lyon.grid5000.fr: enp5s0f0
Prod interface for gros-99.nancy.grid5000.fr: eno1
Prod interface for gros-97.nancy.grid5000.fr: eno1
Prod interface for ecotype-9.nantes.grid5000.fr: eno1
Prod interface for gros-93.nancy.grid5000.fr: eno1
Prod interface for parasilo-4.rennes.grid5000.fr: eno1
Prod interface for ecotype-8.nantes.grid5000.fr: eno1
Prod interface for gros-98.nancy.grid5000.fr: eno1
Prod interface for parasilo-5.rennes.grid5000.fr: eno1
Prod interface for gros-94.nancy.grid5000.fr: eno1
Prod interface for nova-7.lyon.grid5000.fr: enp5s0

Finished 1 tasks (cmd) on {'chirop-5.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

...

Output()

Created 250 namespaces for client_2 (with gateway 10.176.0.1)
gateway_ip=10.140.0.1 for client client_3


Finished 1 tasks (create_macvlan_namespaces) on {'nova-4.lyon.grid5000.fr', 
'nova-5.lyon.grid5000.fr', 'nova-9.lyon.grid5000.fr', 'nova-8.lyon.grid5000.fr', 
'nova-23.lyon.grid5000.fr', 'nova-19.lyon.grid5000.fr', 'nova-22.lyon.grid5000.fr', 
'nova-6.lyon.grid5000.fr', 'nova-7.lyon.grid5000.fr', 'nova-21.lyon.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Created 250 namespaces for client_3 (with gateway 10.140.0.1)


### Setting up GRE tunnels between routers in different clusters

This step will create GRE tunnels between each pair of routers as defined in the topology file. The endpoints of the tunnels use the production IP of the nodes.

In [5]:
experiment.setup_gre_tunnels()

Output()

Link 0: gre1(router_server, 192.168.0.1) <-> gre1(router_client_0, 192.168.0.2)
Link 1: gre2(router_client_0, 192.168.0.5) <-> gre1(router_client_1, 192.168.0.6)
Link 2: gre3(router_client_0, 192.168.0.9) <-> gre1(router_client_3, 192.168.0.10)
Link 3: gre2(router_client_1, 192.168.0.13) <-> gre1(router_client_2, 192.168.0.14)


Finished 1 tasks (setup_gre_router_server) on {'chirop-4.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

...

Output()

Created 1 GRE tunnels on router_client_3 (nova-18.lyon.grid5000.fr)


Finished 1 tasks (setup_gre_router_client_2) on {'ecotype-43.nantes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Created 1 GRE tunnels on router_client_2 (ecotype-43.nantes.grid5000.fr)
Router tunnels: {'router_server': [{'iface': 'gre1', 'ip': '192.168.0.1', 'network': '192.168.0.0', 'tunnel_subnet': IPv4Network('192.168.0.0/30')}], 'router_client_0': [{'iface': 'gre1', 'ip': '192.168.0.2', 'network': '192.168.0.0', 'tunnel_subnet': IPv4Network('192.168.0.0/30')}, {'iface': 'gre2', 'ip': '192.168.0.5', 'network': '192.168.0.4', 'tunnel_subnet': IPv4Network('192.168.0.4/30')}, {'iface': 'gre3', 'ip': '192.168.0.9', 'network': '192.168.0.8', 'tunnel_subnet': IPv4Network('192.168.0.8/30')}], 'router_client_1': [{'iface': 'gre1', 'ip': '192.168.0.6', 'network': '192.168.0.4', 'tunnel_subnet': IPv4Network('192.168.0.4/30')}, {'iface': 'gre2', 'ip': '192.168.0.13', 'network': '192.168.0.12', 'tunnel_subnet': IPv4Network('192.168.0.12/30')}], 'router_client_3': [{'iface': 'gre1', 'ip': '192.168.0.10', 'network': '192.168.0.8', 'tunnel_subnet': IPv4Network('192.168.0.8/30')}], 'router_client_2': [{'ifac

#### FRRouting setup

With GRE tunnels setup between routers, we can now configure and start FRRouting. The frr configuration template defined in the topology file will be used as a base.

In [6]:
experiment.frrouting_setup()
experiment.setup_default_routes()

Output()

Finished 1 tasks (restart_frr_chirop-4.lille.grid5000.fr) on {'chirop-4.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[router_server] chirop-4.lille.grid5000.fr  prod=10.136.0.1  loopback=10.136.3.254  gateway=172.16.47.254  tunnels=1


Output()

...

[router_client_2] ecotype-43.nantes.grid5000.fr  prod=10.176.0.1  loopback=10.176.3.254  gateway=172.16.207.254  tunnels=1


Output()

Finished 1 tasks (restart_frr_nova-18.lyon.grid5000.fr) on {'nova-18.lyon.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[router_client_3] nova-18.lyon.grid5000.fr  prod=10.140.0.1  loopback=10.140.3.254  gateway=172.16.63.254  tunnels=1
172.16.33.4
172.16.66.9
Unknown role: relay_0
172.16.97.25
Unknown role: relay_1
172.16.193.43
Unknown role: relay_2
172.16.52.18
Unknown role: relay_3
setting default routes on 41 nodes
default via 172.16.79.254 dev eno1
gros-97.nancy.grid5000.fr's default route is 172.16.66.9 on eno1
default via 172.16.79.254 dev eno1
gros-99.nancy.grid5000.fr's default route is 172.16.66.9 on eno1
default via 172.16.79.254 dev eno1
gros-90.nancy.grid5000.fr's default route is 172.16.66.9 on eno1
default via 172.16.47.254 dev ens10f0np0
chirop-5.lille.grid5000.fr's default route is 172.16.33.4 on ens10f0np0
default via 172.16.79.254 dev eno1
gros-96.nancy.grid5000.fr's default route is 172.16.66.9 on eno1
default via 172.16.79.254 dev eno1
gros-93.nancy.grid5000.fr's default route is 172.16.66.9 on eno1
default via 172.16.79.254 dev eno1
gros-91.nancy.grid5000.fr's default route is 172

### Upload binary files over to nodes

We build the executables locally first

In [22]:
# TODO: change this path to your project
!cd ../../../g5k_mcast_eval && cargo build --release

   --> /home/corentin/fcquic_applications_master_thesis/multicast-quic/octets/src/lib.rs:474:22
    |
474 |     pub fn get_bytes(&mut self, len: usize) -> Result<Octets> {
    |                      ^^^^^^^^^                        ^^^^^^ the same lifetime is hidden here
    |                      |
    |                      the lifetime is elided here
    |
    = help: the same lifetime is referred to in inconsistent ways, making the signature confusing
    = note: `#[warn(mismatched_lifetime_syntaxes)]` on by default
help: use `'_` for type paths
    |
474 |     pub fn get_bytes(&mut self, len: usize) -> Result<Octets<'_>> {
    |                                                             ++++

   --> /home/corentin/fcquic_applications_master_thesis/multicast-quic/octets/src/lib.rs:491:26
    |
491 |     pub fn get_bytes_mut(&mut self, len: usize) -> Result<OctetsMut> {
    |                          ^^^^^^^^^                        ^^^^^^^^^ the same lifetime is hidden here
    | 

Then we push them to the nodes

In [23]:
import enoslib as en

experiment.push_binaries(
    # TODO: change these paths with the path to your binaries and certificates
    bin_dir="../../../g5k_mcast_eval/target/release",
    cert_dir="../../../g5k_mcast_eval",
    relay_binaries=(), # if you don't have relays (which are other nodes), pass a tuple like as done here 
)

res = en.run_command(
    "sysctl -w net.core.rmem_default=26214400 && sysctl -w net.core.rmem_max=26214400",
    roles=experiment.roles,
)
print("errors: " + str([out.stderr for out in res.filter(status=en.STATUS_FAILED)]))

Output()

Finished 2 tasks (file,copy) on {'nova-23.lyon.grid5000.fr', 'nova-19.lyon.grid5000.fr', 
'gros-98.nancy.grid5000.fr', 'gros-97.nancy.grid5000.fr', 'ecotype-6.nantes.grid5000.fr', 
'parasilo-28.rennes.grid5000.fr', 'gros-93.nancy.grid5000.fr', 
'parasilo-3.rennes.grid5000.fr', 'nova-7.lyon.grid5000.fr', 'ecotype-44.nantes.grid5000.fr', 
'gros-91.nancy.grid5000.fr', 'nova-9.lyon.grid5000.fr', 'gros-95.nancy.grid5000.fr', 
'ecotype-45.nantes.grid5000.fr', 'ecotype-5.nantes.grid5000.fr', 'gros-90.nancy.grid5000.fr',
'parasilo-27.rennes.grid5000.fr', 'ecotype-9.nantes.grid5000.fr', 
'ecotype-46.nantes.grid5000.fr', 'parasilo-7.rennes.grid5000.fr', 
'parasilo-9.rennes.grid5000.fr', 'nova-22.lyon.grid5000.fr', 'chirop-5.lille.grid5000.fr', 
'nova-6.lyon.grid5000.fr', 'ecotype-47.nantes.grid5000.fr', 'ecotype-7.nantes.grid5000.fr', 
'nova-4.lyon.grid5000.fr', 'nova-5.lyon.grid5000.fr', 'parasilo-6.rennes.grid5000.fr', 
'parasilo-26.rennes.grid5000.fr', 'nova-8.lyon.grid5000.fr', 'gros-94.nancy.grid5000.fr', 
'ecotype-48.nantes.grid5000.fr', 'gros-96.nancy.grid5000.fr', 
'parasilo-5.rennes.grid5000.fr', 'parasilo-8.rennes.grid5000.fr', 
'gros-92.nancy.grid5000.fr', 'gros-99.nancy.grid5000.fr', 'parasilo-4.rennes.grid5000.fr', 
'ecotype-8.nantes.grid5000.fr', 'nova-21.lyon.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Pushed ['server', 'client'] to 41 server/client node(s)


Finished 1 tasks (sysctl -w net.core.rmem_default=26214400 && sysctl -w 
net.core.rmem_max=26214400) on {'ecotype-43.nantes.grid5000.fr', 'nova-18.lyon.grid5000.fr', 
'nova-23.lyon.grid5000.fr', 'nova-19.lyon.grid5000.fr', 'gros-98.nancy.grid5000.fr', 
'gros-97.nancy.grid5000.fr', 'ecotype-6.nantes.grid5000.fr', 
'parasilo-28.rennes.grid5000.fr', 'gros-9.nancy.grid5000.fr', 'gros-93.nancy.grid5000.fr', 
'nova-7.lyon.grid5000.fr', 'parasilo-3.rennes.grid5000.fr', 'ecotype-44.nantes.grid5000.fr', 
'gros-91.nancy.grid5000.fr', 'nova-9.lyon.grid5000.fr', 'gros-95.nancy.grid5000.fr', 
'ecotype-45.nantes.grid5000.fr', 'chirop-4.lille.grid5000.fr', 
'ecotype-5.nantes.grid5000.fr', 'gros-90.nancy.grid5000.fr', 
'parasilo-27.rennes.grid5000.fr', 'parasilo-25.rennes.grid5000.fr', 
'ecotype-9.nantes.grid5000.fr', 'ecotype-46.nantes.grid5000.fr', 
'parasilo-7.rennes.grid5000.fr', 'parasilo-9.rennes.grid5000.fr', 'nova-22.lyon.grid5000.fr',
'chirop-5.lille.grid5000.fr', 'nova-6.lyon.grid5000.fr', 'ecotype-47.nantes.grid5000.fr', 
'ecotype-7.nantes.grid5000.fr', 'nova-4.lyon.grid5000.fr', 'nova-5.lyon.grid5000.fr', 
'parasilo-6.rennes.grid5000.fr', 'parasilo-26.rennes.grid5000.fr', 'nova-8.lyon.grid5000.fr',
'gros-94.nancy.grid5000.fr', 'ecotype-48.nantes.grid5000.fr', 'gros-96.nancy.grid5000.fr', 
'parasilo-5.rennes.grid5000.fr', 'parasilo-8.rennes.grid5000.fr', 
'gros-92.nancy.grid5000.fr', 'gros-99.nancy.grid5000.fr', 'parasilo-4.rennes.grid5000.fr', 
'ecotype-8.nantes.grid5000.fr', 'nova-21.lyon.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

errors: []


### Running the relay experiment

Experiment.py provides some basic blocks that should (ideally) allow you to define your own custom experiments.
Below you will find the code for the evaluation of two types of Flexicast QUIC relays, this should hopefully provide enough information.


In [24]:
import concurrent.futures
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Literal
import time

import enoslib as en

from gem.experiment import (
    EvalConfig,
    MetricSpec,
    collect_results,
    run_eval,
)
from gem.remote import (
    run_cmd_bg_enos,
    run_cmd_ssh_parallel,
    send_pkill_hosts,
    ssh_bg_hosts,
)


@dataclass
class RunConfig:
    additional_data_size: int
    test_length: int


@dataclass
class CatEvalConfig(EvalConfig):
    ready_sleep_clients: int = 2
    post_test_buffer: int = 7
    bin_log_level: str = "info"
    cert_path: str = "/tmp"
    server_bin: str = "/tmp/bin/server"
    client_bin: str = "/tmp/bin/client"
    remote_log_root: str = "/tmp/logs"
    num_ns_per_client: int = experiment.topology.netns_per_client
    cc_algo: str = "cubic"
    # cc_algo: str = "reno"
    fallback_delay: int = 10000
    server_cpus: str = "0-7"  # taskset -c range for server
    per_cluster_results: bool = True
    flow_control: int = 16_000_000  # 16 Megabytes
    fc_timer: int = 0

# Metrics can be output by the clients (so application level metrics), they must be output on stdout (not stderr, so using a logger may not work, make sure to print them out) 
# define the results to extract from the client logs, one csv output file is emitted for each metric
METRICS = [
    MetricSpec(
        key="LATENCY",
        column="y_LATENCY",
        pattern=rf"^RESULT-LATENCY-\S+\s+([0-9.]+)\s*$",
    ),
    #  you can add more result types here, e.g.:
    # MetricSpec(key="THROUGHPUT", column="y_THROUGHPUT"),
]

We can now define specific tests based on our `RunConfig`.

For the network categorization test, we simply send one packet of increasing size, and we wait until it has been received by all clients.
- 1KB will fit inside of one packet
- 10KB will fit in one flight of packets
- for larger values, the sender will have to grow its congestion window to be able to send it

In [25]:
def categorization_matrix():
    return [
        RunConfig(additional_data_size=sz, test_length=length)
        # for sz in (10_000, 100_000, 1_000_000, 10_000_000)
        # for sz in (1_000,)
        # for sz in (1_000_000,)
        # for sz in (10_000_000,)
        # for sz, length in ((10_000_000, 20),)
        for sz, length in (
            # (1_000, 20),
            (10_000, 15),
            (100_000, 20),
            (1_000_000, 20),
            (10_000_000, 30),
        )
    ]

To enable us to have graphs that show certain metrics per cluster, we need to pass in a list of the cluster names and their subnets to the clients. Here we construct the lists to pass to the clients

In [19]:
import ipaddress
import pickle
from pathlib import Path

TOPOLOGY_CACHE = Path("./topology_cache.pkl")


# ---------- cluster names & subnets ----------
# the subnets are in experiment.networks, but i need the subnets keyed by their index and not their cluster
def compute_topology_cache():
    site_subnets = {
        site.name: {
            "cluster": site.cluster,
            "num_clients": site.num_machines,
            # NOTE: we only remove the prefix because the arg is an IPv4Addr (in rust), not a network  # "10.x.y.0"
            "subnet": str(experiment.networks[f"subnet_client_{i}"][0].network),
        }
        for i, site in enumerate(experiment.topology.sites)
    }
    return {
        "server_subnet": str(experiment.networks["subnet_server"][0].network),
        "site_subnets": site_subnets,
        "cluster_names": [site.name for site in experiment.topology.sites],
        "cluster_subnets": [
            str(ipaddress.ip_network(info["subnet"]).network_address)
            for info in site_subnets.values()
        ],
        "site_subnets_str": [info["subnet"] for info in site_subnets.values()],
    }


try:
    cache = compute_topology_cache()
    with open(TOPOLOGY_CACHE, "wb") as f:
        pickle.dump(cache, f)
    print(f"saved topology values to {TOPOLOGY_CACHE}")
except (NameError, AttributeError, KeyError):
    print(f"no running experiment, loading topology values from {TOPOLOGY_CACHE}")
    with open(TOPOLOGY_CACHE, "rb") as f:
        cache = pickle.load(f)

server_subnet = cache["server_subnet"]
site_subnets = cache[
    "site_subnets"
]  # NOTE: "subnet" is now a string, e.g. "10.x.y.0/22"
cluster_names = cache["cluster_names"]
cluster_subnets = cache["cluster_subnets"]
site_subnets_str = cache["site_subnets_str"]

print(f"site_subnets_str: {site_subnets_str}")
print(f"cluster_names:  {cluster_names}")
print(f"cluster_subnets: {cluster_subnets}")

saved topology values to topology_cache.pkl
site_subnets_str: ['10.144.0.0/22', '10.158.4.0/22', '10.176.0.0/22', '10.140.0.0/22']
cluster_names:  ['nancy', 'rennes', 'nantes', 'lyon']
cluster_subnets: ['10.144.0.0', '10.158.4.0', '10.176.0.0', '10.140.0.0']


Now that the experiment is defined, we need to specify the commands that will be ran on the nodes.


In [26]:
# ---------- command builders ----------
from gem.cpuload import collect_cpuload, install_cpuload
import subprocess
from pathlib import Path

def download_server_logs(cfg, server_host, run_dir, test_name):
    local_dir = (
        Path(cfg.local_out_dir) / "raw" / test_name / Path(run_dir).name / "server"
    )
    local_dir.mkdir(parents=True, exist_ok=True)

    # `-f` so re-running a collection over an already compressed run is fine
    en.run_command(
        f"gzip -f {run_dir}/server/server.stderr {run_dir}/server/server.stdout "
        f"2>/dev/null; true",
        roles=[server_host],
        task_name="gzip_server_log",
        on_error_continue=True,
        gather_facts=False,
    )

    # the dir also holds the cpuload csv, and everything in it is small once
    # the two logs are compressed, so we just take all of it
    subprocess.run(
        [
            "rsync",
            "-az",
            "-e",
            "ssh -o StrictHostKeyChecking=no -o BatchMode=yes -o LogLevel=ERROR",
            f"root@{server_host.address}:{run_dir}/server/",
            f"{local_dir}/",
        ],
        check=False,
    )

    # immediate verdict: how many clients the server actually accepted
    log = local_dir / "server.stderr.gz"
    if log.exists():
        import gzip as _gzip
        import re as _re

        ids, problems = [], {}
        patterns = (
            "Packet is not Initial",
            "Invalid address validation token",
            "Parsing packet header failed",
            "send() would block",
            "recv() failed",
            "Too many open files",
            "panicked",
        )
        with _gzip.open(log, "rt", errors="replace") as f:
            for line in f:
                m = _re.search(r"I give client_id=(\d+)", line)
                if m:
                    ids.append(int(m.group(1)))
                for p in patterns:
                    if p in line:
                        problems[p] = problems.get(p, 0) + 1
        print(
            f"  [server] accepted {len(ids)} clients"
        )
        for p, n in problems.items():
            print(f"    !! {n}x {p!r}")

    return local_dir


def server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts):
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/server"
    return (
        f"mkdir -p {qlog} && ulimit -n 1048576 && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} taskset -c {cfg.server_cpus} {cfg.server_bin} "
        f"--cert-path {cfg.cert_path} --src {server_ip}:4433 --mc-src-addr {server_ip}:4443 "
        f"--flexicast --fc-timer {cfg.fc_timer} --fall-back-delay {cfg.fallback_delay} "
        f"--unicast --fec-scheduler noredundancy --length {length} "
        f"--cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo} "
        f"--additional-data-size {rc.additional_data_size} --test-start-ts {sleep_deadline_ts} "
        f"--initial-fc-flow {cfg.flow_control} "
    )


# IMPORTANT NOTE: since we have multiple network namespaces defined on each client machine,
# we can run processes in these namespaces using the naming scheme "client-$NS_IDX" (with NS_IDX going from the number of 0 to NSs)
def client_loop_cmd(cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts):
    qlog_base = f"{run_dir}/qlog/client"
    per_cluster_res = "--per-cluster-results" if cfg.per_cluster_results else ""
    return f"""
mkdir -p {run_dir}/client
pids=()
for NS_IDX in $(seq $(( {cfg.num_ns_per_client} - 1 )) -1 0); do
    GLOBAL_IDX=$(( {node_id} * {cfg.num_ns_per_client} + NS_IDX ))
    CLIENT_ID=$(( GLOBAL_IDX + 1 ))
    NS_NAME="client-$NS_IDX"
    mkdir -p {qlog_base}_$CLIENT_ID
    CLIENT_IP=$(ip netns exec $NS_NAME ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1)
    (
    exec ip netns exec $NS_NAME env QLOGDIR={qlog_base}_$CLIENT_ID RUST_LOG_STYLE=never RUST_BACKTRACE=full RUST_LOG={cfg.bin_log_level} \\
        {cfg.client_bin} --server-ip {server_ip} --port 4433 \\
        -l $CLIENT_IP --flexicast -u CLIENT$CLIENT_ID --length {rc.test_length} \\
        --test-start-ts {sleep_deadline_ts} \\
        --additional-data-size {rc.additional_data_size} --cc-algorithm {cfg.cc_algo} --flow-control {cfg.flow_control}  \\
        {per_cluster_res} \\
         {" ".join(f"--cluster-names={name}" for name in cluster_names)} \\
         {" ".join(f"--cluster-subnets={subnet}" for subnet in cluster_subnets)} \\
        > {run_dir}/client/client_$CLIENT_ID.stdout \\
        2> {run_dir}/client/client_$CLIENT_ID.stderr < /dev/null < /dev/null
    ) &
    pids+=($!)
done
for pid in "${{pids[@]}}"; do wait $pid; done
"""


# command to force an NTP sync 
NTP_SYNC_CMD = (
    "timedatectl set-ntp true 2>/dev/null || true; "
    "systemctl enable --now systemd-timesyncd 2>/dev/null || true; "
    "systemctl restart systemd-timesyncd 2>/dev/null || true; "
    "chronyc -a makestep 2>/dev/null || true"
)


def sync_ntp(hosts):
    """Sync the clock on every remote host and on the local machine, so the
    shared test start deadline is computed against a common clock."""
    import subprocess

    run_cmd_ssh_parallel(
        NTP_SYNC_CMD, hosts, check=False, label="ntp-sync", quiet=True
    )
    subprocess.run(NTP_SYNC_CMD, shell=True, check=False)


# ---------- one run of the relay experiment ----------
def run_once(cfg, rc, run_index, test_name):
    """Run one iteration: start the server, start the clients in
    their namespaces, wait for the test to finish, then collect the results."""
    roles_dict = experiment.roles
    node_ips = experiment.node_ips

    server_ip = node_ips["server"][0]
    # NOTE: very important, make sure that this run_id is the same as the one in the SQLOG collection cell below
    run_id = f"sz{rc.additional_data_size}_r{run_index}"
    run_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}"

    # make sure that each client is root because it has to start the clients in network namespaces
    client_hosts = [
        en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
        for h in roles_dict["client"]
    ]
    all_hosts = roles_dict["server"] + client_hosts

    # create the dirs on all of the hosts and stop anything left over from a
    # previous run
    run_cmd_ssh_parallel(
        f"mkdir -p {run_dir}/server {run_dir}/client {run_dir}/qlog ; "
        f"pkill -9 server || true ; pkill -9 client || true",
        all_hosts,
    )
    time.sleep(1)

    # sync NTP time on the server, the clients and the local machine before
    # computing the shared start deadline
    sync_ntp(all_hosts)

    # pick a timestamp in 5 seconds, we pass this to all of the clients that will all wait until that timestamp is reached before starting
    datetime_now = datetime.now()
    sleep_deadline = datetime_now + timedelta(seconds=3)
    sleep_deadline_ts = sleep_deadline.timestamp()


    # start server in bg
    run_cmd_bg_enos(
        server_cmd(cfg, rc, server_ip, run_dir, sleep_deadline_ts),
        roles_dict["server"],
        stdout=f"{run_dir}/server/server.stdout",
        stderr=f"{run_dir}/server/server.stderr",
        task_name="server",
    )
    print("Started server")

    # start CPU load monitor (server only)
    if cfg.monitor_cpu:
        cpuload_py = f"{run_dir}/server/cpuload.py"
        cpuload_csv = f"{run_dir}/server/cpuload.csv"
        install_cpuload(roles_dict["server"], cpuload_py)

        run_cmd_bg_enos(
            f"python3 -u {cpuload_py} {cpuload_csv} {rc.test_length + 5} 0 {cfg.cpu_max}",
            roles_dict["server"],
            stdout=f"{run_dir}/server/cpuload.stdout",
            stderr=f"{run_dir}/server/cpuload.stderr",
            task_name="start_cpuload",
        )

    # start all clients at once, in a single ansible run: each host gets its own
    # command, and they all wait for sleep_deadline_ts so they start together
    ssh_bg_hosts(
        [
            (
                h,
                client_loop_cmd(
                    cfg, rc, server_ip, run_dir, node_id, sleep_deadline_ts
                ),
                f"{run_dir}/client/loop_{node_id}.stdout",
                f"{run_dir}/client/loop_{node_id}.stderr",
            )
            for node_id, h in enumerate(client_hosts)
        ],
        task_name="clients",
    )
    print("Started clients")


    # wait for test duration to pass
    time.sleep(rc.test_length + cfg.post_test_buffer)

    send_pkill_hosts(all_hosts, ["server", "client"])

    time.sleep(1)

    results = collect_results(cfg, client_hosts, run_dir, test_name, METRICS)
    download_server_logs(cfg, roles_dict["server"][0], run_dir, test_name)
    cpu_samples = (
        collect_cpuload(cfg, roles_dict["server"][0], run_dir, test_name)
        if cfg.monitor_cpu
        else []
    )

    return results, cpu_samples


Lauching the test

In [27]:
import logging

logging.getLogger("paramiko").setLevel(logging.WARNING)

N_RUNS = 10

cfg = CatEvalConfig(n_runs=N_RUNS, monitor_cpu=True)
now = datetime.now().strftime("%d-%m-%H-%M%p")

test_name = f"categorization_{now}"
matrix = categorization_matrix()


def row_fields(rc):
    # columns identifying each run in the result CSVs
    return {
        "ADDITIONAL_DATA_SIZE": rc.additional_data_size,
    }


run_eval(
    matrix,
    cfg,
    run_once=run_once,
    test_name=test_name,
    row_fields=row_fields,
    metrics=METRICS,
)

=> {'ADDITIONAL_DATA_SIZE': 10000} run=0 (attempt 1)


  [cmd] done on 41/41 host(s) in 1.6s
  [server] started on 1/1 host(s) in 0.4s
Started server
  [cmd] done on 1/1 host(s) in 0.4s
  [start_cpuload] started on 1/1 host(s) in 0.4s
  [clients] started on 40/40 host(s) in 1.4s
Started clients


Output()

Finished 1 tasks (gzip_server_log) on {'chirop-5.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

...

Output()

Finished 1 tasks (gzip_server_log) on {'chirop-5.lille.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

  [server] accepted 1000 clients
-> collected {'LATENCY': 1000} samples, 15996 cpu samples


Test finished in 2944.616379737854 seconds
Writing CSV file...
wrote npf-out/categorization_22-09-09-35AM.csv (40000 rows)
Writing CSV file...
wrote npf-out/categorization_22-09-09-35AM_cpu.csv (223170 rows)


{'LATENCY': PosixPath('npf-out/categorization_22-09-09-35AM.csv')}

### Downloading SQLOGs from server and clients 



In [29]:
import shutil
import subprocess
from pathlib import Path

import enoslib as en

# uses cfg and test_name from the launching cell above
local_base = Path(f"./sqlogs/{test_name}")
local_base.mkdir(parents=True, exist_ok=True)

server_hosts = experiment.roles["server"]
client_hosts = [
    en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
    for h in experiment.roles["client"]
]
remote_test_dir = f"{cfg.remote_log_root}/{test_name}"


def free_space_gb() -> float:
    return shutil.disk_usage(local_base).free / 1e9


gzip_all_cmd = (
    "find {dir} -name '*.sqlog' -type f -print0 "
    "| xargs -0 -r -n 20 -P $(nproc) gzip -f"
)

print(f"free space: {free_space_gb():.2f} GB")

raw_left = sum(1 for _ in local_base.rglob("*.sqlog"))
if raw_left:
    print(f"compressing {raw_left} already downloaded sqlog file(s)...")
    subprocess.run(
        gzip_all_cmd.format(dir=str(local_base)),
        shell=True,
        check=False,
    )
    print(f"after compressing the local sqlogs: {free_space_gb():.2f} GB free")

gz_res = en.run_command(
    gzip_all_cmd.format(dir=remote_test_dir),
    roles=experiment.roles["client"] + server_hosts,
    on_error_continue=True,
    task_name="gzip_sqlogs",
    gather_facts=False,
)
failed = list(gz_res.filter(status=en.STATUS_FAILED)) + list(
    gz_res.filter(status=en.STATUS_UNREACHABLE)
)
if failed:
    print(f"warning: could not gzip the sqlogs on {len(failed)} host(s)")

try:
    du_res = en.run_command(
        f"du -sb {remote_test_dir} 2>/dev/null | cut -f1",
        roles=experiment.roles["client"] + server_hosts,
        on_error_continue=True,
        task_name="sqlog_size",
        gather_facts=False,
    )
    sizes = []
    for out in du_res:
        try:
            sizes.append(int(out.stdout.strip().splitlines()[0]))
        except (AttributeError, IndexError, ValueError):
            pass
    if sizes:
        print(
            f"compressed sqlogs on the nodes: {sum(sizes) / 1e9:.2f} GB "
            f"(free space: {free_space_gb():.2f} GB)"
        )
    else:
        print("could not read the compressed sqlog size from the nodes")
except Exception as exc:
    print(f"could not compute the size of the remote sqlogs: {exc}")

for host in client_hosts + server_hosts:
    print(f"downloading sqlogs from {host.address}")
    proc = subprocess.run(
        [
            "rsync",
            "-a",
            "--include=*/",
            "--include=*.sqlog.gz",
            "--include=*.sqlog",
            "--exclude=*",
            "--prune-empty-dirs",
            f"root@{host.address}:{remote_test_dir}/",
            f"{local_base}/",
        ],
        check=False,
    )
    if proc.returncode != 0:
        print(f"warning: rsync exited with code {proc.returncode} for {host.address}")

print(f"after the download: {free_space_gb():.2f} GB free")

for qlog_dir in sorted(local_base.glob("*/qlog")):
    for client_dir in sorted(qlog_dir.iterdir()):
        target = qlog_dir.parent / client_dir.name
        # if the same results were already downloaded before, we replace the prev copy
        if target.exists():
            shutil.rmtree(target)
        client_dir.rename(target)
    qlog_dir.rmdir()

print(f"results: {local_base}")

free space: 15.47 GB


Output()

Finished 1 tasks (gzip_sqlogs) on {'nova-23.lyon.grid5000.fr', 'nova-19.lyon.grid5000.fr', 
'gros-98.nancy.grid5000.fr', 'gros-97.nancy.grid5000.fr', 'ecotype-6.nantes.grid5000.fr', 
'parasilo-28.rennes.grid5000.fr', 'gros-93.nancy.grid5000.fr', 
'parasilo-3.rennes.grid5000.fr', 'nova-7.lyon.grid5000.fr', 'ecotype-44.nantes.grid5000.fr', 
'gros-91.nancy.grid5000.fr', 'nova-9.lyon.grid5000.fr', 'gros-95.nancy.grid5000.fr', 
'ecotype-45.nantes.grid5000.fr', 'ecotype-5.nantes.grid5000.fr', 'gros-90.nancy.grid5000.fr',
'parasilo-27.rennes.grid5000.fr', 'ecotype-9.nantes.grid5000.fr', 
'ecotype-46.nantes.grid5000.fr', 'parasilo-7.rennes.grid5000.fr', 
'parasilo-9.rennes.grid5000.fr', 'nova-22.lyon.grid5000.fr', 'chirop-5.lille.grid5000.fr', 
'nova-6.lyon.grid5000.fr', 'ecotype-47.nantes.grid5000.fr', 'ecotype-7.nantes.grid5000.fr', 
'nova-4.lyon.grid5000.fr', 'nova-5.lyon.grid5000.fr', 'parasilo-6.rennes.grid5000.fr', 
'parasilo-26.rennes.grid5000.fr', 'nova-8.lyon.grid5000.fr', 'gros-94.nancy.grid5000.fr', 
'ecotype-48.nantes.grid5000.fr', 'gros-96.nancy.grid5000.fr', 
'parasilo-5.rennes.grid5000.fr', 'parasilo-8.rennes.grid5000.fr', 
'gros-92.nancy.grid5000.fr', 'gros-99.nancy.grid5000.fr', 'parasilo-4.rennes.grid5000.fr', 
'ecotype-8.nantes.grid5000.fr', 'nova-21.lyon.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

...

downloading sqlogs from parasilo-8.rennes.grid5000.fr


downloading sqlogs from parasilo-28.rennes.grid5000.fr


downloading sqlogs from ecotype-45.nantes.grid5000.fr


downloading sqlogs from parasilo-3.rennes.grid5000.fr


after the download: 11.02 GB free
results: sqlogs/categorization_22-09-09-35AM


### Merging SQLOG files together

In [30]:
from pathlib import Path
import sys
import re
import csv
import gzip
import ipaddress
import multiprocessing as mp
import statistics
import orjson


site_subnet_to_cluster = {
    ipaddress.ip_network(subnet, strict=False): name
    for subnet, name in zip(site_subnets_str, cluster_names)
}


def open_sqlog(path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path)


def cluster_from_sqlog(sqlog: Path) -> str | None:
    match = re.search(r"client-Client-(\d+\.\d+\.\d+\.\d+)\.sqlog(?:\.gz)?", sqlog.name)
    if not match:
        return None

    client_ip = ipaddress.ip_address(match.group(1))
    for subnet, cluster in site_subnet_to_cluster.items():
        if client_ip in subnet:
            return cluster

    return None


# the congestion window value is in the recovery:metrics_updated
def extract_congestion_window(sqlog, msg_size, writer, run_index):
    with open_sqlog(sqlog) as src:
        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"recovery:metrics_updated"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":5165.1006,"name":"recovery:metrics_updated","data":{"smoothed_rtt":333.0,"rtt_variance":166.5,"congestion_window":13500,"ssthresh":18446744073709551615}}
            if event.get("name") == "recovery:metrics_updated":
                congestion_window = event.get("data", {}).get("congestion_window")
                if congestion_window is not None:
                    writer.writerow(
                        [event.get("time"), run_index, msg_size, congestion_window]
                    )


def load_fc_sent_times(sqlog):
    sent: dict[int, float] = {}

    with open_sqlog(sqlog) as src:
        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"transport:packet_sent"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                continue

            if event.get("name") == "transport:packet_sent":
                data = event.get("data", {})
                packet_number = (data.get("header") or {}).get("packet_number")
                if packet_number is not None:
                    sent.setdefault(packet_number, event.get("time"))

    return sent


def extract_uc_retransmissions(sqlog, writer, run_index, msg_size, fc_sent):
    fc_recv: dict[int, float] = {}
    fallback_times = []

    with open_sqlog(sqlog) as src:
        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"transport:packet_received"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            if event.get("name") != "transport:packet_received":
                continue

            data = event.get("data", {})
            frames = data.get("frames", []) or []
            has_source_symbol_frame = any(
                frame.get("frame_type") == "unknown"
                and frame.get("raw_frame_type") == 246
                for frame in frames
            )

            for frame in frames:
                if frame.get("frame_type") != "stream" or frame.get("stream_id") != 3:
                    continue

                if has_source_symbol_frame:
                    # received on the fc flow: remember its arrival time
                    packet_number = data.get("header").get("packet_number")
                    fc_recv.setdefault(packet_number, event.get("time"))
                else:
                    # fallback retransmission received on the unicast connection
                    fallback_times.append(event.get("time"))

        if not fallback_times:
            return

        # compute the 
        diffs = [
            fc_recv[packet_number] - fc_sent[packet_number]
            for packet_number in fc_recv
            if packet_number in fc_sent
        ]
        if len(diffs) < 20:
            # not enough fc packets seen to estimate the clock offset reliably
            return

        clock_offset = statistics.median(diffs)
        for time in fallback_times:
            writer.writerow([run_index, msg_size, time - clock_offset])


# NOTE: for the smoothed RTT extract data.smoothed_rtt from recovery:metrics_updated records
def extract_smoothed_rtt(sqlog, cluster, msg_size, writer):
    with open_sqlog(sqlog) as src:
        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"recovery:metrics_updated"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":30.884829,"name":"recovery:metrics_updated","data":{"min_rtt":15.767142,"smoothed_rtt":15.767142,"latest_rtt":15.767142,"rtt_variance":7.883571,"bytes_in_flight":0}}
            if event.get("name") == "recovery:metrics_updated":
                smoothed_rtt = event.get("data", {}).get("latest_rtt")
                if smoothed_rtt is not None:
                    writer.writerow(
                        [event.get("time"), cluster, msg_size, smoothed_rtt]
                    )


# to compute the download completion time, take the time from the first stream frame with stream ID 3 and the last, subtract end from start
def extract_download_completion_time(sqlog, cluster, msg_size, writer, run_index):
    with open_sqlog(sqlog) as src:
        start_frame_time = None
        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"transport:packet_received"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # Small packets: entire message contained in one QUIC packet, contains FIN flag
            #   {"time":5074.468,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":2},"raw":{"length":1085,"payload_length":1068},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":0,"length":1036,"fin":true}]}}
            # large packet:
            # here is one packet that contains part of the message:
            # {"time":5072.8867,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":3},"raw":{"length":1284,"payload_length":1267},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":1235,"length":1234}]}}
            # here is the final packet (with fin flag raised)
            # {"time":5307.8716,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":92},"raw":{"length":635,"payload_length":618},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":99453,"length":583,"fin":true}]}}
            if event.get("name") == "transport:packet_received":
                frames = event.get("data", {}).get("frames", []) or []
                for frame in frames:
                    if (
                        frame.get("frame_type") == "stream"
                        and frame.get("stream_id") == 3
                    ):
                        fin = frame.get("fin", False)
                        # length = frame.get("length")
                        if fin:
                            end_time = event.get("time")
                            if start_frame_time is None:
                                writer.writerow(
                                    [
                                        cluster,
                                        run_index,
                                        msg_size,
                                        end_time,
                                        end_time,
                                        0,
                                    ]
                                )
                                break
                            else:
                                writer.writerow(
                                    [
                                        cluster,
                                        run_index,
                                        msg_size,
                                        start_frame_time,
                                        end_time,
                                        (end_time - start_frame_time),
                                    ]
                                )
                                break
                        else:
                            if start_frame_time is None:
                                start_frame_time = event.get("time")


# Source - https://stackoverflow.com/a/16974075
# Retrieved 2026-09-16, License - CC BY-SA 3.0
def missing_elements(L):
    start, end = min(L), max(L)
    return sorted(set(range(start, end + 1)).difference(L))


# the quiche version used does not skip packet numbers, so we can store all of the packet numbers we've seen on the FC flow and search for hole to find losses
# NOTE: FCQUIC only sends a SourceSymbol frame (FEC) for packets sent on the FC flow, so here since I can't obtain the path ID of a packet, i'll kinda hack
# my way around it by only storing the packet numbers of the packets that contained a stream frame and a source symbol frame, so that I can know which packets
# from the FC flow i've seen and which were lost
def extract_losses(sqlog, cluster, msg_size, writer, run_id, run_index):
    with open_sqlog(sqlog) as src:
        packets_seen: set[int] = set()

        for line in src:
            line = line.strip()
            if not line:
                continue

            if '"transport:packet_received"' not in line:
                continue

            try:
                event = orjson.loads(line)
            except ValueError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":5179.9014,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":30},"raw":{"length":1284,"payload_length":1267},"frames":[{"frame_type":"unknown","raw_frame_type":246},{"frame_type":"stream","stream_id":3,"offset":31490,"length":1232}]}}
            if event.get("name") == "transport:packet_received":
                data = event.get("data", {})
                frames = data.get("frames", []) or []
                has_source_symbol_frame = False

                for frame in frames:
                    frame_type = frame.get("frame_type")

                    # frames with value 246 (0xF6) are SourceSymbol (FEC) frames
                    if frame_type == "unknown" and frame.get("raw_frame_type") == 246:
                        has_source_symbol_frame = True

                    if (
                        frame_type == "stream"
                        and frame.get("stream_id") == 3
                        # and has_source_symbol_frame
                    ) or frame_type == "ping":
                        packet_num = (data.get("header") or {}).get("packet_number")
                        if packet_num is not None:
                            packets_seen.add(packet_num)

        all_packet_nums = sorted(packets_seen)
        if len(all_packet_nums) > 0:
            losses = missing_elements(all_packet_nums)
            if len(losses) != 0:
                total_lost = len(losses)
                # total packets sent on the FC flow = packets seen + packets lost in between
                total_sent = len(all_packet_nums) + total_lost
                loss_rate = total_lost / total_sent
                writer.writerow(
                    [cluster, run_index, msg_size, total_lost, total_sent, loss_rate]
                )


class RowList(list):

    def writerow(self, row):
        self.append(row)


def parse_sqlog_file(trace):
    file, run_index, msg_size = trace
    cluster = cluster_from_sqlog(file)
    if cluster is None:
        return file, None

    rtt, dl, loss, uc_retrans = RowList(), RowList(), RowList(), RowList()
    extract_smoothed_rtt(file, cluster, msg_size, rtt)
    extract_download_completion_time(file, cluster, msg_size, dl, run_index)
    extract_losses(
        file, cluster, msg_size, loss, f"sz{msg_size}_r{run_index}", run_index
    )
    extract_uc_retransmissions(
        file, uc_retrans, run_index, msg_size, fc_sent_times[(run_index, msg_size)]
    )
    return file, (rtt, dl, loss, uc_retrans)


def parse_server_sqlog_file(trace):
    file, run_index, msg_size = trace

    cwnd = RowList()
    extract_congestion_window(file, msg_size, cwnd, run_index)
    return file, cwnd


local_base = Path(f"./sqlogs/{test_name}")

trace_files = []
server_fc_trace = []
for run_conf in matrix:
    for run_index in range(N_RUNS):
        run_id = f"sz{run_conf.additional_data_size}_r{run_index}"

        run_dir = local_base / run_id

        # the qlogs are stored in one dir per client, and the file name contains the client's IP
        for file in sorted(run_dir.glob("client_*/client-*.sqlog*")):
            # we keep the message size of the run next to each file so it can be 
            # written as a column in the csvs below
            trace_files.append((file, run_index, run_conf.additional_data_size))

        for file in sorted(run_dir.glob("server/server-fc-*.sqlog*")):
            server_fc_trace.append((file, run_index, run_conf.additional_data_size))


if not trace_files:
    print(f"no client sqlog files found in {local_base}")

if not server_fc_trace:
    print(f"no server fc sqlog files found in {local_base}")


# the fc sent time is used to align the client times with the server's fc time
fc_sent_times = {
    (run_index, msg_size): load_fc_sent_times(file)
    for file, run_index, msg_size in server_fc_trace
}

est_rtt_csv = Path(f"./npf-out/") / test_name / f"est_rtt.csv"
dl_completion_csv = Path(f"./npf-out/") / test_name / f"dl_completion.csv"
losses_csv = Path(f"./npf-out/") / test_name / f"losses.csv"
uc_retransmissions_csv = Path(f"./npf-out/") / test_name / f"uc_retransmissions.csv"
cwnd_csv = Path(f"./npf-out/") / test_name / f"cwnd.csv"
est_rtt_csv.parent.mkdir(parents=True, exist_ok=True)

with (
    open(est_rtt_csv, "w", newline="") as rtt_out,
    open(dl_completion_csv, "w", newline="") as dl_out,
    open(losses_csv, "w", newline="") as losses,
    open(cwnd_csv, "w", newline="") as cwnd,
    open(uc_retransmissions_csv, "w", newline="") as uc_retr,
):
    rtt_writer = csv.writer(rtt_out)
    dl_writer = csv.writer(dl_out)
    losses_writer = csv.writer(losses)
    cwnd_writer = csv.writer(cwnd)
    uc_ret_writer = csv.writer(uc_retr)

    rtt_writer.writerow(["time", "cluster", "msg_size", "smoothed_rtt"])
    cwnd_writer.writerow(["time", "run_index", "msg_size", "cwnd"])
    dl_writer.writerow(["cluster", "run_index", "msg_size", "start", "end", "duration"])
    losses_writer.writerow(
        ["cluster", "run_index", "msg_size", "lost", "total_msg_packets", "loss_rate"]
    )
    uc_ret_writer.writerow(["run_index", "msg_size", "time"])

    with mp.get_context("fork").Pool() as pool:
        for file, rows in pool.imap(parse_sqlog_file, trace_files):
            if rows is None:
                print(f"couldn't find the cluster of {file}, skipping")
                continue
            rtt_writer.writerows(rows[0])
            dl_writer.writerows(rows[1])
            losses_writer.writerows(rows[2])
            uc_ret_writer.writerows(rows[3])

    with mp.get_context("fork").Pool() as pool:
        for file, cwnd_row in pool.imap(parse_server_sqlog_file, server_fc_trace):
            cwnd_writer.writerows(cwnd_row)

print(f"estimated RTT csv: {est_rtt_csv}")
print(f"download completion csv: {dl_completion_csv}")
print(f"losses csv: {losses_csv}")
print(f"cwnd csv: {cwnd_csv}")
print(f"uc fallback retransmissions csv: {uc_retransmissions_csv}")


estimated RTT csv: npf-out/categorization_22-09-09-35AM/est_rtt.csv
download completion csv: npf-out/categorization_22-09-09-35AM/dl_completion.csv
losses csv: npf-out/categorization_22-09-09-35AM/losses.csv
cwnd csv: npf-out/categorization_22-09-09-35AM/cwnd.csv
uc fallback retransmissions csv: npf-out/categorization_22-09-09-35AM/uc_retransmissions.csv


### Graphing the results


In [40]:
import subprocess
from pathlib import Path

test_name = "categorization_21-09-10-02AM"

NO_TITLE = True
out_path = f"./graphs/{test_name}"
output_path = Path(out_path)
output_path.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "./mcast_graphs.py",
        f"./npf-out/raw/{test_name}",
        out_path,  # out path
        test_name,
        f"./npf-out/{test_name}/est_rtt.csv",  # estimated rtt csv
        f"./npf-out/{test_name}/dl_completion.csv",  # download completion time csv
        f"./npf-out/{test_name}/losses.csv",  # losses csv
        f"./npf-out/{test_name}/cwnd.csv",  # cwnd csv
        f"./npf-out/{test_name}/uc_retransmissions.csv",  # uc retransmissions csv
        # "--cpu-path",
        # f"./npf-out/{test_name}_cpu.csv",
        *(
            ["--no-title"] if NO_TITLE else []
        ),  # list unpacking, this avoids the empty ""
    ],
    check=True,
)

wrote ./graphs/categorization_21-09-10-02AM/rct_vs_runs.svg
wrote ./graphs/categorization_21-09-10-02AM/rct_variance_across_runs.svg
wrote ./graphs/categorization_21-09-10-02AM/rct_boxplot.svg


/home/corentin/fcquic_applications_master_thesis/evaluations/g5k_expe_example/network_cat/./mcast_graphs.py:340: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.
  g = sns.lineplot(


wrote ./graphs/categorization_21-09-10-02AM/losses_vs_cluster.svg
wrote ./graphs/categorization_21-09-10-02AM/est_rtt_vs_size_cluster.svg
Fastest req. comp. time run=9, slowest rct run=1
wrote ./graphs/categorization_21-09-10-02AM/cwnd_growth_100000.svg 
Fastest req. comp. time run=4, slowest rct run=8
wrote ./graphs/categorization_21-09-10-02AM/cwnd_growth_1000000.svg 
Fastest req. comp. time run=3, slowest rct run=4
wrote ./graphs/categorization_21-09-10-02AM/cwnd_growth_10000000.svg 
additional data size 10000: 3000 samples
  Lyon: 750 samples, median 8.11 ms
  Nancy: 750 samples, median 4.38 ms
  Nantes: 750 samples, median 12.37 ms
  Rennes: 750 samples, median 11.25 ms
wrote ./graphs/categorization_21-09-10-02AM/cdf_categorization_21-09-10-02AM_datasize_10000.svg
additional data size 100000: 3000 samples
  Lyon: 750 samples, median 180.10 ms
  Nancy: 750 samples, median 176.88 ms
  Nantes: 750 samples, median 184.54 ms
  Rennes: 750 samples, median 183.94 ms
wrote ./graphs/catego

CompletedProcess(args=['./mcast_graphs.py', './npf-out/raw/categorization_21-09-10-02AM', './graphs/categorization_21-09-10-02AM', 'categorization_21-09-10-02AM', './npf-out/categorization_21-09-10-02AM/est_rtt.csv', './npf-out/categorization_21-09-10-02AM/dl_completion.csv', './npf-out/categorization_21-09-10-02AM/losses.csv', './npf-out/categorization_21-09-10-02AM/cwnd.csv', './npf-out/categorization_21-09-10-02AM/uc_retransmissions.csv', '--no-title'], returncode=0)

In [ ]:
# test_name="categorization_21-09-10-02AM"

subprocess.run(
    [
        "./uc_stats.py",
        test_name,
        "--by-cluster",
    ],
    check=True,
)

#### Compressing the csv results

In [ ]:
import subprocess

# test_name="categorization_21-09-10-43AM"
# test_name="categorization_21-09-13-25PM"
# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
        f"./npf-out/{test_name}/dl_completion.csv",
        f"./npf-out/{test_name}/cwnd.csv",
        f"./npf-out/{test_name}/uc_retransmissions.csv",
        f"./npf-out/{test_name}/est_rtt.csv",
        f"./npf-out/{test_name}/losses.csv",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
        f"./topology_cache.pkl",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./graphs/{test_name}/{test_name}.tar.gz",
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
    ],
    check=True,
)
subprocess.run(
    [
        "rm",
        "-rf",
        f"./npf-out/{test_name}/dl_completion.csv",
        f"./npf-out/{test_name}/est_rtt.csv",
        f"./npf-out/{test_name}/cwnd.csv",
        f"./npf-out/{test_name}/uc_retransmissions.csv",
        f"./npf-out/{test_name}/losses.csv",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

CompletedProcess(args=['rm', '-rf', './npf-out/categorization_22-09-09-35AM/dl_completion.csv', './npf-out/categorization_22-09-09-35AM/est_rtt.csv', './npf-out/categorization_22-09-09-35AM/cwnd.csv', './npf-out/categorization_22-09-09-35AM/uc_retransmissions.csv', './npf-out/categorization_22-09-09-35AM/losses.csv', './npf-out/raw/categorization_22-09-09-35AM/', './sqlogs/categorization_22-09-09-35AM/'], returncode=0)

Opposite code to unarchive the results, in order to regenerate graphs if needed

In [38]:
import subprocess
from pathlib import Path

test_name = "categorization_21-09-10-02AM"

archive_path = Path(f"./graphs/{test_name}/{test_name}.tar.gz")

if archive_path.exists():
    print(f"decompressing {archive_path}...")

    Path("./npf-out/").mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "tar",
            "xzf",
            str(archive_path),
            "-C",
            "./",
        ],
        check=True,
    )
    print(f"decompressed files to ./npf-out/ and ./sqlogs/")
else:
    print(f"Couldn't find: {archive_path}")

decompressing graphs/categorization_21-09-10-02AM/categorization_21-09-10-02AM.tar.gz...
decompressed files to ./npf-out/ and ./sqlogs/


#### Deleting log files from all clusters

In [ ]:
import subprocess
from pathlib import Path

remote_log_root = "/tmp/logs"
matrix = categorization_matrix()
for run_conf in matrix:

    remote_qlog_dir = f"{remote_log_root}/{test_name}/"

    en.run_command(
        f"rm -rf {remote_qlog_dir}",
        roles=experiment.roles["client"] + experiment.roles["server"],
    )

print("done deleting sqlog files")

## Important: Stopping the current booking
Always, always stop your booking if you are done earlier.

In [37]:
experiment.stop_reservation()

INFO     [G5k] Reloading 2209758 from lille                              ]8;id=412852;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=483906;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 2068926 from lyon                               ]8;id=544339;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=466773;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 6936367 from nancy                              ]8;id=920160;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=680318;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 338507 from nantes                              ]8;id=881812;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=830448;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 4126993 from rennes                             ]8;id=229767;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=246493;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

...

INFO     [G5k] Killing the job (nantes, 338507)                          ]8;id=261416;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=769412;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#278\278]8;;\

INFO     [G5k] Job killed (nantes, 338507)                               ]8;id=686062;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=416200;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#259\259]8;;\

INFO     [G5k] Killing the job (rennes, 4126993)                         ]8;id=376752;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=13276;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#278\278]8;;\

INFO     [G5k] Job killed (rennes, 4126993)                              ]8;id=76606;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=686349;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#259\259]8;;\

Reservation stopped.
